# Обучение Ember-NNUE на nnue-pytorch

Этот ноутбук обучает **внешнюю сеть для движка Ember** в бинарном формате контейнера
V2 (магия `0x6A448AFA`):

| Параметр            | Значение                                  |
|---------------------|-------------------------------------------|
| `Full_Threats`      | 60 720 входов, веса i8                    |
| `HalfKAv2_hm^` (PSQ)| 22 528 входов, веса i16                   |
| `L1` (hidden)       | 1024 (SCReLU)                             |
| стеки `Affine`      | 8 бакетов × (32 → 32 → 1), SCCReLU        |
| PSQT-бакетов        | 8                                         |
| ARCH_HASH           | `0x0256acdf`                              |
| FT_HEADER_HASH      | `0x6165ddc9`                              |
| STACK_HASH          | `0x63337116`                              |

Датасет — [nodes5000pv2_UHO.binpack](https://huggingface.co/datasets/official-stockfish/master-binpacks/blob/main/nodes5000pv2_UHO.binpack)
(~40 ГБ, sha256 `7a80e6d2…bdc34`). Ноутбук сам качает его, ставит зависимости,
собирает C++ data-loader, обучает и конвертирует чекпоинт в `.nnue` для Ember.

**Порядок запуска**
1. Runtime → Change runtime type → **GPU** (T4 / A100 / L4) и ~70+ ГБ диска.
2. Запускай все клетки по порядку. Скачивание данных займёт время.
3. Один «супербатч» = 100 000 000 позиций = 6104 батча × 16384. По умолчанию
   300 супербатчей, авто-сохранение каждые **3**.
4. При обрыве сессии перезапусти с 11-й клетки — обучение продолжится с последнего
   чекпоинта (чекпоинты на Google Drive).


## 0. Окружение и GPU


In [ ]:
!nvidia-smi
import torch, sys
print("Python", sys.version)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. nnue-pytorch (закреплённый коммит)

Используется коммит `a7830b2a91d15f6d3214bd21b1a6cc5cf7701b82` — последний перед
добавлением `PP_3Wide`. В нём `Full_Threats` = 60 720 входов и дефолт
`Full_Threats+HalfKAv2_hm^`, то есть ровно та архитектура, которую ждёт Ember.


In [ ]:
%%bash
set -e
cd /content
if [ ! -d /content/nnue-pytorch/.git ]; then
  echo "Cloning nnue-pytorch..."
  git clone https://github.com/official-stockfish/nnue-pytorch /content/nnue-pytorch
fi
cd /content/nnue-pytorch
git fetch --quiet --all 2>/dev/null || true
git checkout -q a7830b2a91d15f6d3214bd21b1a6cc5cf7701b82
git rev-parse HEAD
echo "OK"


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq >/dev/null 2>&1
apt-get install -y -qq cmake g++ >/dev/null 2>&1
pip install -q -r /content/nnue-pytorch/requirements.txt
pip uninstall -y -q tensorflow tensorflow-cpu tf-nightly tf-keras 2>/dev/null || true
python -c "import torch, tyro, lightning; print('torch', torch.__version__, '| tyro OK | lightning', lightning.__version__)"


### Сборка C++ data-loader (обязательно)

Без `libtraining_data_loader.so` тренировка не стартует — она линкуется через
ctypes из `./build/`, поэтому тренировать нужно из каталога `/content/nnue-pytorch`.


In [ ]:
%%bash
set -e
cd /content/nnue-pytorch
cmake -S data_loader/cpp -B build -DCMAKE_BUILD_TYPE=Release > /tmp/cmake_cfg.log 2>&1
cmake --build build -j"$(nproc)" > /tmp/cmake_build.log 2>&1
ls -la build/libtraining_data_loader* build/training_data_loader* 2>/dev/null | head
cd /content/nnue-pytorch && python -c "import data_loader; print('data_loader OK')"


## 2. Датасет `nodes5000pv2_UHO.binpack` (~40 ГБ)

Файл качается в `/content/data` (диск рантайма). Клетка докачивает с места обрыва.
Ожидаемый размер: **40 292 454 358** байт (проверено по карточке файла на HuggingFace).


In [ ]:
%%bash
set -e
URL="https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/nodes5000pv2_UHO.binpack"
DATASET="/content/data/nodes5000pv2_UHO.binpack"
EXPECTED=40292454358
mkdir -p /content/data
for i in $(seq 1 12); do
  HAVE=$(stat -c %s "$DATASET" 2>/dev/null || echo 0)
  if [ "$HAVE" -eq "$EXPECTED" ]; then echo "complete ($HAVE bytes)"; break; fi
  echo "attempt $i: ${HAVE} / ${EXPECTED}"
  curl -L -C - -sS --retry 3 --retry-delay 5 -o "$DATASET" "$URL" || true
  sleep 2
done
SIZE=$(stat -c %s "$DATASET")
echo "size=$SIZE expected=$EXPECTED"
if [ "$SIZE" -eq "$EXPECTED" ]; then echo "DATASET_READY"; else echo "DATASET_INCOMPLETE -> rerun this cell"; fi


## 3. Обучение

**Супербатч = одна эпоха.** Параметры меняются в клетке ниже. Обучение пишет через
Lightning чекпоинты в `OUT_ROOT` на Google Drive; при повторном запуске само подхватит
последний `checkpoints/last.ckpt` (`--resume-from-checkpoint`).

По умолчанию: 300 супербатчей, авто-сейв каждые **3**, батч 16384, эпоха = 100М позиций,
оптимизатор rangerlite (стандартный пайплайн SF), loss WDL-ремаппинг как в конфиге.


In [ ]:
import os

os.environ['NNUE_REPO']   = '/content/nnue-pytorch'
os.environ['DATASET']     = '/content/data/nodes5000pv2_UHO.binpack'
os.environ['DRIVE_DIR']   = '/content/drive/MyDrive/Ember_Networks'
os.environ['RUN_NAME']    = 'ember_ft_hm_2026'
os.environ['MAX_EPOCHS']  = '300'
os.environ['SAVE_EVERY']  = '3'
os.environ['BATCH_SIZE']  = '16384'
os.environ['EPOCH_SIZE']  = '100000000'
os.environ['NUM_WORKERS'] = '4'
os.environ['SMOKE']       = 'False'

print("config:", {k: os.environ[k] for k in ['NNUE_REPO','DATASET','DRIVE_DIR','RUN_NAME','MAX_EPOCHS','SAVE_EVERY','BATCH_SIZE','EPOCH_SIZE','NUM_WORKERS']})


In [ ]:
import os, sys, glob, signal, subprocess

os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
os.environ["CUDNN_DETERMINISTIC"] = "0"
os.environ["CUDNN_BENCHMARK"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PL_TORCH_BACKEND"] = "torch"
os.environ["LIGHTNING_PRECISION"] = "16-mixed"

NNUE_REPO = os.environ['NNUE_REPO']
DATASET   = os.environ['DATASET']
DRIVE_DIR = os.environ['DRIVE_DIR']
RUN_NAME  = os.environ['RUN_NAME']
OUT_ROOT  = f"{DRIVE_DIR}/{RUN_NAME}"

max_epochs   = int(os.environ['MAX_EPOCHS'])
save_every   = int(os.environ['SAVE_EVERY'])
batch_size   = int(os.environ['BATCH_SIZE'])
epoch_size   = int(os.environ['EPOCH_SIZE'])
num_workers  = int(os.environ['NUM_WORKERS'])
smoke        = os.environ.get('SMOKE', 'False').strip().lower() in ('1','true','yes')

assert os.path.exists(DATASET), "dataset not found, run download cell first"

if smoke:
    max_epochs = 2
    save_every = 1
    epoch_size = min(epoch_size, 500_000)
    num_workers = 2
    print(">>> SMOKE TEST: 2 superbatches of", epoch_size, "positions")

os.makedirs(OUT_ROOT, exist_ok=True)

ckpts = sorted(glob.glob(f"{OUT_ROOT}/**/checkpoints/last.ckpt", recursive=True),
               key=os.path.getmtime)
resume = ckpts[-1] if ckpts else None

cmd = [
    sys.executable, "train.py", DATASET,
    "--features", "Full_Threats+HalfKAv2_hm^",
    "--l1", "1024", "--l2", "32", "--l3", "32",
    "--batch-size", str(batch_size),
    "--epoch-size", str(epoch_size),
    "--max-epochs", str(max_epochs),
    "--network-save-period", str(save_every),
    "--save-top-k", "-1",
    "--num-workers", str(num_workers),
    "--default-root-dir", OUT_ROOT,
    "--accelerator", "cuda",
]
if resume:
    cmd += ["--resume-from-checkpoint", resume]
    print(">>> resume from", resume)
print("$", " ".join(cmd), "\n")

process = subprocess.Popen(
    cmd,
    cwd=NNUE_REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True,
    env=os.environ.copy(),
    start_new_session=True,
)

def terminate_process_group(process):
    if process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait()

try:
    for line in process.stdout:
        print(line, end='')
except KeyboardInterrupt:
    terminate_process_group(process)
    raise

return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)